In [1]:
import json
import time

import faiss
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer


/Users/aaravmenon/Aalen/database/project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1030 12:33:30.644000 64490 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [3]:
files = ["naruto_s1_backup.json", "naruto_s2.json", "naruto_s2_secondpart.json", "naruto_s2_thirdpart.json"]

documents = []
ids = []
metadata = []
full_dataset = []  # Added to track dictionaries for both files cleanly

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        current_season = json.load(f)

    documents.extend([doc["raw_text"] for doc in current_season])
    ids.extend([str(doc["id"]) for doc in current_season])
    metadata.extend([{"url": doc["url"]} for doc in current_season])
    full_dataset.extend(current_season)  # Keep the dictionaries aligned with the index

print(f"Loaded {len(documents)} total documents into memory.")

Loaded 720 total documents into memory.


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

document_embeddings = model.encode(
    documents,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype("float32")

print("Embedding shape:", document_embeddings.shape)


Batches: 100%|██████████| 23/23 [00:08<00:00,  2.66it/s]

Embedding shape: (720, 384)


In [6]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")

if "naruto_documents" in {
    collection.name for collection in chroma_client.list_collections()
}:
    chroma_client.delete_collection("naruto_documents")

chroma_collection = chroma_client.create_collection(
    name="naruto_documents",
    configuration={"hnsw": {"space": "cosine"}},
)

chroma_collection.add(
    ids=ids,
    embeddings=document_embeddings.tolist(),
    documents=documents,
    metadatas=metadata,
)

print(f"Stored {chroma_collection.count()} documents in ChromaDB")


Stored 720 documents in ChromaDB


In [7]:
faiss_index = faiss.IndexFlatIP(document_embeddings.shape[1])
faiss_index.add(document_embeddings)

print(f"Stored {faiss_index.ntotal} documents in FAISS")


Stored 720 documents in FAISS


In [8]:
def chroma_search(query, top_k=5):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    result = chroma_collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    return [
        {
            "id": result["ids"][0][rank],
            "score": 1 - result["distances"][0][rank],
            "url": result["metadatas"][0][rank]["url"],
            "text": result["documents"][0][rank],
        }
        for rank in range(len(result["ids"][0]))
    ]


In [9]:
def faiss_search(query, top_k=5):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, top_k)

    return [
        {
            "id": ids[index],
            "score": float(score),
            "url": full_dataset[index]["url"],  # Fixed: Looks up the aligned dictionary list
            "text": full_dataset[index]["raw_text"],  # Fixed: Looks up the aligned dictionary list
        }
        for score, index in zip(scores[0], indices[0])
        if 0 <= index < len(ids)  # Double-sided bounds safety check
    ]

In [10]:
def print_results(name, results):
    print(f"\n{name}")
    for rank, result in enumerate(results, start=1):
        print(f"\n{rank}. Score: {result['score']:.4f}")
        print("URL:", result["url"])
        print("Text:", result["text"][:300])


In [12]:
print("FAISS vectors:", faiss_index.ntotal)
print("Dataset documents:", len(full_dataset))
print("IDs:", len(ids))

FAISS vectors: 720
Dataset documents: 720
IDs: 720


In [ ]:
query = "Nine tail fox attack"

print_results("ChromaDB results", chroma_search(query))
print_results("FAISS exact results", faiss_search(query))


In [ ]:
from benchmark import run_benchmark

run_benchmark()
